In [1]:
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

1.0

In [2]:
DB_FILE = Path("data/data.db")
assert DB_FILE.exists(), f"Database not found at {DB_FILE}. Copy data.db into data/ first!"

with sqlite3.connect(DB_FILE) as conn:
    df_raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
    df_ref = pd.read_sql_query("SELECT * FROM ohlc", conn)

print(f"Raw shape: {df_raw.shape} | Reference shape: {df_ref.shape}")

Raw shape: (397711, 8) | Reference shape: (368251, 8)


1.1 - Initial Inspection & Ticker Normalization

In [3]:
# 1.1 Basic Inspection
display(df_raw.info())
display(df_raw.describe())

def standardize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Audit column for original ticker
    out["raw_ticker"] = out["ticker"].astype(str)
    # Strip HTML/XML tags, whitespace, and uppercase
    out["ticker"] = (
        out["ticker"]
        .astype(str)
        .str.replace(r"<[^>]+>", "", regex=True)
        .str.strip()
        .str.upper()
    )
    out["ts"] = pd.to_datetime(out["ts"], errors="coerce").dt.normalize()
    
    num_cols = ["open", "high", "low", "close", "adj_close", "volume"]
    for col in num_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

df_work = standardize_dataframe(df_raw)
df_ref_clean = standardize_dataframe(df_ref)

print("Date span:", df_work["ts"].min().date(), "to", df_work["ts"].max().date())
print(f"Unique raw tickers: {df_work['raw_ticker'].nunique()} -> Normalized: {df_work['ticker'].nunique()}")
print("Tickers:", sorted(df_work["ticker"].unique()))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


None

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


Date span: 1962-01-02 to 2024-12-06
Unique raw tickers: 150 -> Normalized: 30
Tickers: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


1.2 — Vectorized Data Quality Detector

In [4]:
def make_issue_rows(df_sub: pd.DataFrame, issue_type: str, desc: str, severity: str) -> pd.DataFrame:
    if df_sub.empty:
        return pd.DataFrame(columns=["ticker", "issue_type", "issue_date", "description", "severity"])
    return pd.DataFrame({
        "ticker": df_sub["ticker"].fillna("UNKNOWN"),
        "issue_type": issue_type,
        "issue_date": df_sub["ts"].dt.strftime("%Y-%m-%d").fillna("UNKNOWN"),
        "description": desc,
        "severity": severity
    })

def run_diagnostics(df: pd.DataFrame, ref_df: pd.DataFrame) -> pd.DataFrame:
    reports = []
    price_cols = ["open", "high", "low", "close", "adj_close"]
    
    # 1. Missing values per column
    for col in price_cols + ["volume"]:
        mask_na = df[col].isna()
        sev = "low" if col == "volume" else "medium"
        reports.append(make_issue_rows(df[mask_na], f"missing_{col}", f"Column '{col}' is NULL/NaN", sev))
        
    # 2. Non-positive or non-finite prices
    for col in price_cols:
        mask_bad = df[col].notna() & (~np.isfinite(df[col]) | (df[col] <= 0))
        reports.append(make_issue_rows(df[mask_bad], f"nonpositive_{col}", f"Column '{col}' <= 0 or infinite", "high"))
        
    # 3. Negative or zero volume
    mask_neg_vol = df["volume"].notna() & (df["volume"] < 0)
    reports.append(make_issue_rows(df[mask_neg_vol], "negative_volume", "Volume < 0", "medium"))
    
    mask_zero_vol = df["volume"].notna() & (df["volume"] == 0)
    reports.append(make_issue_rows(df[mask_zero_vol], "zero_volume", "Volume == 0 (potential stale trade)", "low"))
    
    # 4. OHLC Inconsistencies (only check where raw OHLC are finite/present)
    valid_ohlc = df[["open", "high", "low", "close"]].notna().all(axis=1)
    
    mask_h_lt_c = valid_ohlc & (df["high"] < df["close"])
    reports.append(make_issue_rows(df[mask_h_lt_c], "high_lt_close", "High < Close (impossible)", "high"))
    
    mask_l_gt_c = valid_ohlc & (df["low"] > df["close"])
    reports.append(make_issue_rows(df[mask_l_gt_c], "low_gt_close", "Low > Close (impossible)", "high"))
    
    mask_open_out = valid_ohlc & ((df["open"] < df["low"]) | (df["open"] > df["high"]))
    reports.append(make_issue_rows(df[mask_open_out], "open_out_of_bounds", "Open outside [Low, High]", "medium"))
    
    mask_low_gt_high = valid_ohlc & (df["low"] > df["high"])
    reports.append(make_issue_rows(df[mask_low_gt_high], "low_gt_high", "Low > High", "high"))

    # 5. Duplicate (ticker, ts) keys
    mask_dups = df.duplicated(subset=["ticker", "ts"], keep=False)
    reports.append(make_issue_rows(df[mask_dups], "duplicate_key", "Duplicate (ticker, ts) record", "medium"))

    # 6. Outliers (> 3 std dev simple returns within ticker)
    df_sorted = df.dropna(subset=["ticker", "ts"]).sort_values(["ticker", "ts"]).copy()
    # Deduplicate temporarily for return screen so 0% duplicate returns don't skew std dev
    df_unique_temp = df_sorted.drop_duplicates(subset=["ticker", "ts"]).copy()
    df_unique_temp["ret"] = df_unique_temp.groupby("ticker")["close"].pct_change(fill_method=None)
    
    stats = df_unique_temp.groupby("ticker")["ret"].agg(
        mean_ret="mean", std_ret=lambda s: s.std(ddof=1)
    ).reset_index()
    df_unique_temp = df_unique_temp.merge(stats, on="ticker", how="left")
    
    mask_outlier = (
        df_unique_temp["ret"].notna() &
        (df_unique_temp["std_ret"] > 0) &
        ((df_unique_temp["ret"] - df_unique_temp["mean_ret"]).abs() > 3 * df_unique_temp["std_ret"])
    )
    reports.append(make_issue_rows(
        df_unique_temp[mask_outlier],
        "extreme_return",
        "Daily simple return exceeds 3 sample std devs from ticker mean",
        "medium"
    ))

    # 7. Missing Trading Days vs Reference Calendar
    missing_day_rows = []
    for tkr, grp in df_sorted.groupby("ticker"):
        ref_dates = pd.DatetimeIndex(ref_df.loc[ref_df["ticker"] == tkr, "ts"].dropna().unique()).sort_values()
        if ref_dates.empty:
            continue
        obs_dates = pd.DatetimeIndex(grp["ts"].unique()).sort_values()
        expected_span = ref_dates[(ref_dates >= obs_dates.min()) & (ref_dates <= obs_dates.max())]
        missing_dates = expected_span.difference(obs_dates)
        for d in missing_dates:
            missing_day_rows.append({
                "ticker": tkr,
                "issue_type": "missing_trading_day",
                "issue_date": d.strftime("%Y-%m-%d"),
                "description": f"Ticker {tkr} missing reference trading session",
                "severity": "medium"
            })
    if missing_day_rows:
        reports.append(pd.DataFrame(missing_day_rows))

    # 8. Corrupted vendor adj_close vs Reference Adjustment Factor
    ref_factors = ref_df.dropna(subset=["ticker", "ts", "close", "adj_close"]).drop_duplicates(["ticker", "ts"]).copy()
    ref_factors["ref_factor"] = ref_factors["adj_close"] / ref_factors["close"]
    merged = df.merge(ref_factors[["ticker", "ts", "ref_factor"]], on=["ticker", "ts"], how="left")
    expected_adj = merged["close"] * merged["ref_factor"]
    mask_bad_adj = (
        merged["adj_close"].notna() &
        expected_adj.notna() &
        (~np.isclose(merged["adj_close"], expected_adj, rtol=1e-3, atol=0.02))
    )
    reports.append(make_issue_rows(
        merged[mask_bad_adj],
        "corrupted_adj_close",
        "Vendor adj_close deviates from reference adjustment factor",
        "high"
    ))

    return pd.concat(reports, ignore_index=True)

dq_report_pre = run_diagnostics(df_work, df_ref_clean)
display(dq_report_pre.groupby(["issue_type", "severity"]).size().reset_index(name="issue_count"))

,issue_type,severity,issue_count
0,corrupted_adj_close,high,11265
1,duplicate_key,medium,58920
2,extreme_return,medium,5056
3,missing_adj_close,medium,19855
4,nonpositive_open,high,5
5,open_out_of_bounds,medium,5
6,zero_volume,low,81


Missing Trading Days Summary & Synthetic Demonstration

In [5]:
# Summary for real dataset
for tkr, grp in df_work.groupby("ticker"):
    span_start = grp["ts"].min().strftime("%Y-%m-%d")
    span_end = grp["ts"].max().strftime("%Y-%m-%d")
    n_miss = len(dq_report_pre[(dq_report_pre["ticker"] == tkr) & (dq_report_pre["issue_type"] == "missing_trading_day")])
    if n_miss > 0:
        print(f"Ticker {tkr} is missing {n_miss} trading days between {span_start} and {span_end}")
print("Real dataset summary: All 30 tickers are missing 0 trading days over their active histories.")

# Synthetic demonstration on a tiny example with 1 removed date & injected errors
synth_ref = pd.DataFrame({
    "ticker": ["DEMO"] * 4,
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]),
    "open": [100.0, 101.0, 102.0, 103.0],
    "high": [105.0, 106.0, 107.0, 108.0],
    "low": [99.0, 100.0, 101.0, 102.0],
    "close": [102.0, 103.0, 104.0, 105.0],
    "adj_close": [102.0, 103.0, 104.0, 105.0],
    "volume": [1000, 1000, 1000, 1000]
})
# Remove 2024-01-03 and inject high < close on 2024-01-04
synth_dirty = synth_ref.iloc[[0, 2, 3]].copy()
synth_dirty.loc[synth_dirty["ts"] == "2024-01-04", "high"] = 95.0

synth_dq = run_diagnostics(synth_dirty, synth_ref)
display(synth_dq)

Real dataset summary: All 30 tickers are missing 0 trading days over their active histories.


,ticker,issue_type,issue_date,description,severity
0,DEMO,high_lt_close,2024-01-04,High < Close (impossible),high
1,DEMO,open_out_of_bounds,2024-01-04,"Open outside [Low, High]",medium
2,DEMO,low_gt_high,2024-01-04,Low > High,high
3,DEMO,missing_trading_day,2024-01-03,Ticker DEMO missing reference trading session,medium


Section 1.3 — Cleaning Pipeline & Quarantine Accounting

In [6]:
def clean_ohlc_pipeline(df_in: pd.DataFrame, ref_df: pd.DataFrame):
    df = standardize_dataframe(df_in)
    quarantined_list = []
    
    # Initialize repair flags
    for col in ["open", "high", "low", "close", "adj_close", "volume"]:
        df[f"{col}_repaired"] = False

    # 1. Drop rows where ALL raw prices are missing
    mask_all_missing = df[["open", "high", "low", "close"]].isna().all(axis=1)
    if mask_all_missing.any():
        q = df[mask_all_missing].copy()
        q["quarantine_reason"] = "All raw OHLC prices missing"
        quarantined_list.append(q)
        df = df[~mask_all_missing].copy()

    # 2. Remove exact duplicates; quarantine conflicting (ticker, ts) keys
    price_vol_cols = ["open", "high", "low", "close", "adj_close", "volume"]
    before_dedup = len(df)
    df = df.drop_duplicates(subset=["ticker", "ts"] + price_vol_cols, keep="first").copy()
    exact_dups_removed = before_dedup - len(df)

    mask_conflict = df.duplicated(subset=["ticker", "ts"], keep=False)
    if mask_conflict.any():
        q = df[mask_conflict].copy()
        q["quarantine_reason"] = "Conflicting values for same (ticker, ts)"
        quarantined_list.append(q)
        df = df[~mask_conflict].copy()

    # Sort strictly by ticker and date before any filling
    df = df.sort_values(["ticker", "ts"]).reset_index(drop=True)

    # 3. Handle non-positive or impossible raw OHLC relationships (Option 1: Set to NaN & forward-fill within ticker)
    for col in ["open", "high", "low", "close"]:
        bad_mask = df[col].isna() | (~np.isfinite(df[col])) | (df[col] <= 0)
        df.loc[bad_mask, col] = np.nan
        df.loc[bad_mask, f"{col}_repaired"] = True

    # Flag impossible bounds (e.g. open outside [low, high], high < close, low > close)
    inv_open = (df["open"] < df["low"]) | (df["open"] > df["high"])
    df.loc[inv_open, "open"] = np.nan
    df.loc[inv_open, "open_repaired"] = True

    inv_hl = (df["high"] < df["close"]) | (df["low"] > df["close"]) | (df["low"] > df["high"])
    df.loc[inv_hl, ["high", "low"]] = np.nan
    df.loc[inv_hl, ["high_repaired", "low_repaired"]] = True

    # Forward fill raw prices within each ticker
    for col in ["open", "high", "low", "close"]:
        df[col] = df.groupby("ticker")[col].ffill()

    # Recheck OHLC envelope after forward-filling (expand high/low if ffill violated today's range)
    row_max = df[["open", "close"]].max(axis=1)
    row_min = df[["open", "close"]].min(axis=1)
    
    fix_h = df["high"] < row_max
    df.loc[fix_h, "high"] = row_max[fix_h]
    df.loc[fix_h, "high_repaired"] = True

    fix_l = df["low"] > row_min
    df.loc[fix_l, "low"] = row_min[fix_l]
    df.loc[fix_l, "low_repaired"] = True

    # 4. Reconstruct missing or corrupted adj_close using date-matched reference factor
    ref_sub = ref_df.dropna(subset=["ticker", "ts", "close", "adj_close"]).drop_duplicates(["ticker", "ts"]).copy()
    ref_sub["trusted_factor"] = ref_sub["adj_close"] / ref_sub["close"]
    df = df.merge(ref_sub[["ticker", "ts", "trusted_factor"]], on=["ticker", "ts"], how="left")

    expected_adj = df["close"] * df["trusted_factor"]
    needs_adj_repair = (
        df["adj_close"].isna() |
        (df["adj_close"] <= 0) |
        df["close_repaired"] |
        (expected_adj.notna() & ~np.isclose(df["adj_close"], expected_adj, rtol=1e-3, atol=0.02))
    )
    can_repair_adj = needs_adj_repair & expected_adj.notna() & (expected_adj > 0)
    df.loc[can_repair_adj, "adj_close"] = expected_adj[can_repair_adj]
    df.loc[can_repair_adj, "adj_close_repaired"] = True
    df.drop(columns=["trusted_factor"], inplace=True)

    # 5. Handle missing or negative volume (fill with 0 and flag)
    bad_vol = df["volume"].isna() | (~np.isfinite(df["volume"])) | (df["volume"] < 0)
    df.loc[bad_vol, "volume"] = 0.0
    df.loc[bad_vol, "volume_repaired"] = True

    # 6. Quarantine any remaining unresolved rows (e.g. leading NaN at start of ticker history)
    price_cols = ["open", "high", "low", "close", "adj_close"]
    unresolved_mask = (
        df[price_cols].isna().any(axis=1) |
        (df[price_cols] <= 0).any(axis=1) |
        (df["low"] > df["open"]) | (df["open"] > df["high"]) |
        (df["low"] > df["close"]) | (df["close"] > df["high"])
    )
    if unresolved_mask.any():
        q = df[unresolved_mask].copy()
        q["quarantine_reason"] = "Leading gap or unresolvable price/factor"
        quarantined_list.append(q)
        df = df[~unresolved_mask].copy()

    # Format ts as ISO string YYYY-MM-DD for SQL compatibility
    df["ts"] = df["ts"].dt.strftime("%Y-%m-%d")
    df_quarantine = pd.concat(quarantined_list, ignore_index=True) if quarantined_list else pd.DataFrame()
    
    stats = {
        "raw_rows": len(df_in),
        "exact_dups_removed": exact_dups_removed,
        "quarantined_rows": len(df_quarantine),
        "cleaned_rows": len(df)
    }
    return df.reset_index(drop=True), df_quarantine, stats

df_cleaned, df_quarantined, clean_stats = clean_ohlc_pipeline(df_raw, df_ref_clean)
print("Cleaning Accounting:", clean_stats)
display(df_quarantined[["ticker", "ts", "open", "high", "low", "close", "quarantine_reason"]])

Cleaning Accounting: {'raw_rows': 397711, 'exact_dups_removed': 29460, 'quarantined_rows': 5, 'cleaned_rows': 368246}


,ticker,ts,open,high,low,close,quarantine_reason
0,CVX,1962-01-02,NaN,3.30000,3.24000,3.30000,Leading gap or unresolvable price/factor
1,JNJ,1962-01-02,NaN,0.22338,0.22222,0.22338,Leading gap or unresolvable price/factor
2,MCD,1966-07-05,NaN,0.27366,0.26749,0.26955,Leading gap or unresolvable price/factor
3,MRK,1962-01-02,NaN,0.39703,0.39317,0.39537,Leading gap or unresolvable price/factor
4,PG,1962-01-02,NaN,1.44000,1.42000,1.43000,Leading gap or unresolvable price/factor


Final Validation & Saving ohlc_hw_cleaned

In [7]:
# Strict Validation Assertions
p_cols = ["open", "high", "low", "close", "adj_close"]
assert not df_cleaned[p_cols].isna().any().any(), "Found NaNs in cleaned prices!"
assert (df_cleaned[p_cols] > 0).all().all(), "Found non-positive prices!"
assert not df_cleaned.duplicated(subset=["ticker", "ts"]).any(), "Found duplicate (ticker, ts)!"
assert (df_cleaned["low"] <= df_cleaned["open"]).all() and (df_cleaned["open"] <= df_cleaned["high"]).all()
assert (df_cleaned["low"] <= df_cleaned["close"]).all() and (df_cleaned["close"] <= df_cleaned["high"]).all()
assert (df_cleaned["volume"] >= 0).all()

with sqlite3.connect(DB_FILE) as conn:
    df_cleaned.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)

print("Successfully validated and wrote 'ohlc_hw_cleaned' to database.")

Successfully validated and wrote 'ohlc_hw_cleaned' to database.
